# Build Ground Truth Datasets

Generate single-food and multi-food datasets with known calories and macros from a nutrition database.

In [26]:
import pandas as pd
import numpy as np
import json, os
from pathlib import Path
from openai import OpenAI

In [27]:
DATA_DIR = Path('data')
SOURCE_DIR = DATA_DIR / 'source'

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

## Load & clean McCance & Widdowson's dataset
Keep only food name, calories, protein, fat, carbs. Trace values → 0.

In [28]:
SRC_FILE = SOURCE_DIR / 'McCance_Widdowsons_Composition_of_Foods_Integrated_Dataset_2021..xlsx'

raw = pd.read_excel(SRC_FILE, sheet_name='1.3 Proximates', header=0, skiprows=[1, 2])
raw.head()

,Food Code,Food Name,Description,Group,Previous,Main data references,Footnote,Water (g),Total nitrogen (g),Protein (g),...,cis-Poly FA /100g Food (g),Poly FA /100g FA (g),Poly FA /100g food (g),Sat FA excl Br /100g FA (g),Sat FA excl Br /100g food (g),Branched chain FA /100g FA (g),Branched chain FA /100g food (g),Trans FAs /100g FA (g),Trans FAs /100g food (g),Cholesterol (mg)
0,13-145,"Ackee, canned, drained",8 cans,DG,554,"MW4, 1978; and Vegetables, Herbs and Spices Su...",NaN,76.7,0.46,2.9,...,0.60,NaN,N,36.50,4.44,0.10,0.01,0.00,0.00,0.0
1,13-146,"Agar, dried",Literature sources,DG,NaN,Wu Leung et al. (1972) Food composition table ...,NaN,9.7,0.26,1.3,...,NaN,NaN,0.40,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,13-147,"Agar, dried, soaked and drained",Literature sources,DG,NaN,Wu Leung et al. (1972) Food composition table ...,NaN,84.2,0.03,0.2,...,NaN,NaN,Tr,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,13-148,"Alfalfa sprouts, raw",Analytical and literature sources,DG,NaN,"Vegetables, Herbs and Spices Supplement, 1991",NaN,93.4,0.64,4.0,...,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,13-801,"Allspice, ground",Literature sources,H,NaN,Marsh et al. (1977) Composition of foods: spic...,NaN,8.5,0.98,6.1,...,NaN,NaN,2.40,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [29]:
KEEP_COLS = {
    'Food Code': 'food_code',
    'Food Name': 'food_name',
    'Protein (g)': 'protein_g',
    'Fat (g)': 'fat_g',
    'Carbohydrate (g)': 'carbs_g',
    'Energy (kcal) (kcal)': 'calories',
}

df = raw[KEEP_COLS.keys()].rename(columns=KEEP_COLS).copy()

# convert numeric cols - 'Tr' (trace) and 'N' become NaN, then we treat trace as 0
num_cols = ['protein_g', 'fat_g', 'carbs_g', 'calories']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

# drop rows with no food name
df = df.dropna(subset=['food_name']).reset_index(drop=True)

print(f'{len(df)} foods loaded')
df.head(10)

2887 foods loaded


,food_code,food_name,protein_g,fat_g,carbs_g,calories
0,13-145,"Ackee, canned, drained",2.9,15.2,0.8,151.0
1,13-146,"Agar, dried",1.3,1.2,0.0,16.0
2,13-147,"Agar, dried, soaked and drained",0.2,0.1,0.0,2.0
3,13-148,"Alfalfa sprouts, raw",4.0,0.7,0.4,24.0
4,13-801,"Allspice, ground",6.1,8.7,0.0,0.0
5,14-870,"Almonds, flaked and ground",21.1,55.8,6.9,612.0
6,14-897,"Almonds, toasted",21.0,52.5,5.9,579.0
7,14-898,"Almonds, weighed with shells",7.8,18.5,1.9,205.0
8,14-896,"Almonds, whole kernels",21.2,49.9,5.3,554.0
9,13-150,"Amaranth leaves, boiled in unsalted water",3.0,0.3,0.3,16.0


## LLM description function
Use an LLM to convert DB food names into natural human-like prompts with realistic weights.

In [30]:
DESCRIBE_SYSTEM = """You are a food portion expert. You will receive a numbered list of food entries. Each entry may contain multiple food components separated by " | " — these are parts of a SINGLE meal and must be described together as one item in the output.

Return a JSON array with EXACTLY one element per numbered entry. Each element has:
- "prompt": a natural, casual description someone would say (e.g. "a grilled chicken breast with a side of rice")
- "weights": list of weights in grams, one per food component within that entry, in the same order

Be realistic with portions. A chicken breast is ~150g, an apple ~180g, a slice of bread ~35g, a bowl of rice ~250g, etc.
Vary the serving sizes naturally - not everything should be one standard portion.

ONLY return the JSON array, nothing else. The array length MUST match the number of entries."""

DESCRIBE_EXAMPLES = """Input foods:
1. Apples, eating, raw, flesh and skin
2. Chicken breast strips, stir-fried | Rice, white, basmati, boiled
3. Cheddar cheese | Bread, white, sliced
4. Salmon, grilled
5. Potatoes, old, baked, flesh and skin | Butter | Cheddar cheese

Output:
[
  {"prompt": "an apple", "weights": [180]},
  {"prompt": "stir fried chicken breast with a bowl of rice", "weights": [150, 250]},
  {"prompt": "a cheese sandwich", "weights": [30, 70]},
  {"prompt": "a grilled salmon fillet", "weights": [120]},
  {"prompt": "a baked potato with butter and cheese", "weights": [300, 15, 40]}
]"""

In [31]:
def describe_foods(food_name_groups: list[list[str]], batch_size: int = 10, model: str = "google/gemini-2.0-flash-001") -> list[dict]:
    """Ask LLM to pick natural descriptions and weights for food groups.
    
    Args:
        food_name_groups: list of lists, e.g. [["Chicken"], ["Rice", "Butter"]]
    Returns:
        list of dicts with 'prompt' and 'weights' keys (None for failed items).
    """
    results = []
    n_batches = (len(food_name_groups) + batch_size - 1) // batch_size

    for i in range(0, len(food_name_groups), batch_size):
        batch = food_name_groups[i:i + batch_size]
        batch_text = "\n".join(f"{j+1}. {' | '.join(group)}" for j, group in enumerate(batch))

        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": DESCRIBE_SYSTEM},
                {"role": "user", "content": DESCRIBE_EXAMPLES.split("Output:")[0].strip()},
                {"role": "assistant", "content": DESCRIBE_EXAMPLES.split("Output:")[1].strip()},
                {"role": "user", "content": f"Input foods:\n{batch_text}"},
            ],
            temperature=0.4,
        )

        text = resp.choices[0].message.content.strip()
        if text.startswith("```"):
            text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            print(f"Warning: batch {i//batch_size} failed to parse, skipping")
            results.extend([None] * len(batch))
            continue

        if len(parsed) != len(batch):
            print(f"Warning: batch {i//batch_size} got {len(parsed)} items for {len(batch)} inputs")
            parsed = parsed[:len(batch)]
            while len(parsed) < len(batch):
                parsed.append(None)

        results.extend(parsed)
        print(f"Batch {i//batch_size + 1}/{n_batches} done")

    return results

## Build dataset

In [32]:
def build_dataset(
    df,
    n: int = 200,
    multi_fraction: float = 0.3,
    soften_descriptions: float = 1.0,
    model: str = "google/gemini-2.0-flash-001",
    name: str = "dataset",
    seed: int = 42,
):
    """Build a benchmark dataset of food prompts with ground truth macros.
    
    Args:
        df: source nutrition dataframe (per 100g values)
        n: number of examples to generate
        multi_fraction: fraction of examples with 2-4 food components (0.0 to 1.0)
        soften_descriptions: fraction to send through LLM for natural descriptions.
            0 or False = all raw "Xg of Y" prompts with random weights.
            1 or True = all LLM-described. Float between = mix.
        model: openrouter model for description generation
        name: dataset name (used for export filename)
        seed: random seed for reproducibility
    """
    rng = np.random.default_rng(seed)
    
    # decide how many components per example
    n_multi = int(n * multi_fraction)
    n_single = n - n_multi
    component_counts = [1] * n_single + [rng.integers(2, 5) for _ in range(n_multi)]
    rng.shuffle(component_counts)
    
    # sample foods for each example
    food_groups = []
    for count in component_counts:
        foods = df.sample(n=count, replace=False, random_state=rng).reset_index(drop=True)
        food_groups.append(foods)
    
    # decide which examples get softened vs raw
    n_soft = int(n * float(soften_descriptions))
    soft_mask = np.array([True] * n_soft + [False] * (n - n_soft))
    rng.shuffle(soft_mask)
    
    # get LLM descriptions for the soft ones
    soft_indices = np.where(soft_mask)[0]
    soft_descriptions = []
    if len(soft_indices) > 0:
        soft_food_names = [food_groups[i]['food_name'].tolist() for i in soft_indices]
        soft_descriptions = describe_foods(soft_food_names, model=model)
    
    # build rows
    rows = []
    soft_idx = 0
    for i in range(n):
        foods = food_groups[i]
        use_soft = soft_mask[i] and soft_idx < len(soft_descriptions)
        desc = soft_descriptions[soft_idx] if use_soft else None
        if use_soft:
            soft_idx += 1
        
        if desc is not None and len(desc.get('weights', [])) == len(foods):
            prompt = desc['prompt']
            weights = desc['weights']
        else:
            # fallback: raw mode with random weights
            weights = list(rng.choice(np.arange(10, 510, 5), size=len(foods)))
            parts = [f"{w}g of {fn}" for w, fn in zip(weights, foods['food_name'])]
            prompt = ", ".join(parts)
        
        # compute ground truth by summing scaled macros
        total = {'calories': 0, 'protein_g': 0, 'fat_g': 0, 'carbs_g': 0}
        components = []
        for j, (_, food) in enumerate(foods.iterrows()):
            w = weights[j]
            scale = w / 100
            total['calories'] += food['calories'] * scale
            total['protein_g'] += food['protein_g'] * scale
            total['fat_g'] += food['fat_g'] * scale
            total['carbs_g'] += food['carbs_g'] * scale
            components.append({'food_name': food['food_name'], 'weight_g': w})
        
        rows.append({
            'prompt': prompt,
            'components': components,
            'calories': round(total['calories'], 1),
            'protein_g': round(total['protein_g'], 1),
            'fat_g': round(total['fat_g'], 1),
            'carbs_g': round(total['carbs_g'], 1),
        })
    
    result = pd.DataFrame(rows)
    
    out_path = DATA_DIR / f'{name}.json'
    result.to_json(out_path, orient='records', indent=2)
    print(f"Saved {len(result)} examples to {out_path}")
    
    return result

In [34]:
# all natural descriptions, 30% multi-food
# important that the model we use here has good benchmark score or else benchmark is not valid - this model needs to understand well relationship between text descriptions and weights
ds = build_dataset(df, n=5000, multi_fraction=0.3, soften_descriptions=0.5, name="benchmark", model="google/gemini-3-flash-preview") 
ds.head(10)

Batch 1/250 done
Batch 2/250 done
Batch 3/250 done
Batch 4/250 done
Batch 5/250 done
Batch 6/250 done
Batch 7/250 done
Batch 8/250 done
Batch 9/250 done
Batch 10/250 done
Batch 11/250 done
Batch 12/250 done
Batch 13/250 done
Batch 14/250 done
Batch 15/250 done
Batch 16/250 done
Batch 17/250 done
Batch 18/250 done
Batch 19/250 done
Batch 20/250 done
Batch 21/250 done
Batch 22/250 done
Batch 23/250 done
Batch 24/250 done
Batch 25/250 done
Batch 26/250 done
Batch 27/250 done
Batch 28/250 done
Batch 29/250 done
Batch 30/250 done
Batch 31/250 done
Batch 32/250 done
Batch 33/250 done
Batch 34/250 done
Batch 35/250 done
Batch 36/250 done
Batch 37/250 done
Batch 38/250 done
Batch 39/250 done
Batch 40/250 done
Batch 41/250 done
Batch 42/250 done
Batch 43/250 done
Batch 44/250 done
Batch 45/250 done
Batch 46/250 done
Batch 47/250 done
Batch 48/250 done
Batch 49/250 done
Batch 50/250 done
Batch 51/250 done
Batch 52/250 done
Batch 53/250 done
Batch 54/250 done
Batch 55/250 done
Batch 56/250 done
B

,prompt,components,calories,protein_g,fat_g,carbs_g
0,a bowl of homemade potato chips,"[{'food_name': 'Potato chips, homemade, fried ...",90.9,1.6,3.0,15.3
1,a serving of pork and chicken chow mein,"[{'food_name': 'Chow mein, pork and chicken, h...",252.0,32.2,6.6,20.3
2,"440g of Lamb, shoulder, whole, roasted, lean a...","[{'food_name': 'Lamb, shoulder, whole, roasted...",1034.0,85.8,77.0,0.0
3,"420g of Chicken, whole, corn-fed, raw, meat an...","[{'food_name': 'Chicken, whole, corn-fed, raw,...",596.4,51.2,43.3,0.0
4,a portion of dry egg pasta,"[{'food_name': 'Pasta, egg, white, dried, raw'...",310.2,11.0,3.1,63.6
5,"420g of Beans, haricot, canned, re-heated, dra...","[{'food_name': 'Beans, haricot, canned, re-hea...",457.0,37.8,5.9,67.5
6,a grilled haddock fillet,"[{'food_name': 'Haddock, flesh only, grilled',...",137.2,33.5,0.4,0.0
7,"30g of Nut roast, homemade","[{'food_name': 'Nut roast, homemade', 'weight_...",100.5,4.0,7.0,5.6
8,"500g of Cake, Swiss rolls, chocolate covered a...","[{'food_name': 'Cake, Swiss rolls, chocolate c...",2070.0,22.5,113.5,255.0
9,"505g of Haddock, flesh only, smoked, steamed","[{'food_name': 'Haddock, flesh only, smoked, s...",479.8,113.6,2.5,0.0
